In [34]:
import numpy as np
from numpy.random import rand
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder

In [35]:
X=np.array(([0.9,0.8],[0.6,0.3],[0.9,0.1],[0.9,0.8],[0.2,0.35],[0.4,0.39]))
Y=np.array((['Netural'],['Positive'],['Positive'],['Netural'],['Negative'],['Negative']))

In [36]:
enc=OneHotEncoder(sparse_output=False,categories='auto')
Y=enc.fit_transform(Y.reshape(len(Y),-1))

In [37]:
enc.categories_

[array(['Negative', 'Netural', 'Positive'], dtype='<U8')]

In [38]:
Y

array([[0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.]])

In [39]:
def sigmoid(z):
  return 1/(1+np.exp(-z))


def softmax(z):
  return np.exp(z)/np.sum(np.exp(z),axis=1,keepdims=True)

class NeuralNetwork:
  def __init__(self,x,y,nodes_in_layer1=4,nodes_in_layer2=3,nodes_in_layer3=3,l_rate=1):

    self.inputs_in_layer0=x
    self.y=y

    self.nodes_in_layer1=nodes_in_layer1
    self.nodes_in_layer2=nodes_in_layer2
    self.nodes_in_layer3=nodes_in_layer3


    self.l_rate=l_rate # Added this line to initialize l_rate

    self.thetas_layer0=np.random.rand(self.inputs_in_layer0.shape[1]+1,self.nodes_in_layer1)
    self.thetas_layer1=np.random.rand(self.nodes_in_layer1 + 1,self.nodes_in_layer2)
    self.thetas_layer2=np.random.rand(self.nodes_in_layer2 + 1,self.nodes_in_layer3)

  def feedforward(self):

    n=self.inputs_in_layer0.shape[0]

    self.Z1=self.thetas_layer0[0]+np.dot(self.inputs_in_layer0,self.thetas_layer0[1:])
    self.layer1=sigmoid(self.Z1)

    self.Z2=self.thetas_layer1[0]+np.dot(self.layer1,self.thetas_layer1[1:])
    self.layer2=sigmoid(self.Z2)

    self.Z3=self.thetas_layer2[0]+np.dot(self.layer2,self.thetas_layer2[1:])
    self.layer3=softmax(self.Z3)

    return self.layer3

  def cost_func(self):

    self.n=self.inputs_in_layer0.shape[0]
    self.cost=(1/self.n)*np.sum(-self.y*np.log(self.layer3))
    return self.cost

  def backprop(self):

    # Output layer (Layer 3) gradients
    self.dE_dZ3 = self.layer3 - self.y # Corrected: dE/dZ3 for softmax+cross-entropy
    self.dE_dtheta2 = np.dot(self.layer2.T, self.dE_dZ3)
    self.dE_dbais2 = np.sum(self.dE_dZ3, axis=0) # Corrected: sum along samples for bias gradient

    # Hidden layer 2 (Layer 2) gradients
    self.dE_dlayer2 = np.dot(self.dE_dZ3, self.thetas_layer2[1:].T)
    self.dE_dZ2 = np.multiply(self.dE_dlayer2, (sigmoid(self.Z2)*(1-sigmoid(self.Z2))))
    self.dE_dtheta1 = np.dot(self.layer1.T, self.dE_dZ2)
    self.dE_dbais1 = np.sum(self.dE_dZ2, axis=0) # Corrected: sum along samples for bias gradient

    # Hidden layer 1 (Layer 1) gradients
    self.dE_dlayer1 = np.dot(self.dE_dZ2, self.thetas_layer1[1:].T)
    self.dE_dZ1 = np.multiply(self.dE_dlayer1, (sigmoid(self.Z1)*(1-sigmoid(self.Z1))))
    self.dE_dtheta0 = np.dot(self.inputs_in_layer0.T, self.dE_dZ1)
    self.dE_dbais0 = np.sum(self.dE_dZ1, axis=0) # Corrected: sum along samples for bias gradient

    # Update weights and biases
    self.thetas_layer2[1:] = self.thetas_layer2[1:] - self.l_rate * self.dE_dtheta2
    self.thetas_layer1[1:] = self.thetas_layer1[1:] - self.l_rate * self.dE_dtheta1
    self.thetas_layer0[1:] = self.thetas_layer0[1:] - self.l_rate * self.dE_dtheta0

    self.thetas_layer2[0] = self.thetas_layer2[0] - self.l_rate * self.dE_dbais2
    self.thetas_layer1[0] = self.thetas_layer1[0] - self.l_rate * self.dE_dbais1
    self.thetas_layer0[0] = self.thetas_layer0[0] - self.l_rate * self.dE_dbais0

    return self

In [40]:
NN.Z3

array([[-8.20084354, 10.32681591,  3.4924267 ],
       [-0.50283675, -1.6399365 ,  5.23315284],
       [-1.5162006 , -0.81372453,  5.6354662 ],
       [-8.20084354, 10.32681591,  3.4924267 ],
       [ 6.33188424, -4.44521531, -0.01277855],
       [ 5.85151528, -4.25841841,  0.36525582]])

In [41]:
NN.layer3

array([[8.97578888e-09, 9.98925023e-01, 1.07496759e-03],
       [3.21398559e-03, 1.03087920e-03, 9.95755135e-01],
       [7.81707939e-04, 1.57806906e-03, 9.97640223e-01],
       [8.97578888e-09, 9.98925023e-01, 1.07496759e-03],
       [9.98226185e-01, 2.08350282e-05, 1.75297984e-03],
       [9.95833446e-01, 4.05040356e-05, 4.12605012e-03]])

In [42]:
NN=NeuralNetwork(X,Y)
epoch=500
losses=[]
for i in range(epoch):
  predicted_output=NN.feedforward()
  error=NN.cost_func()
  losses.append(error)
  u=NN.backprop()
  print("iteration # ",i+1)
  print("Actual Output: \n",Y)
  print("Predicted Output: \n",predicted_output,"\n")
  print("cost: \n",error,"\n")

Streaming output truncated to the last 5000 lines.
cost: 
 1.1039576795285435 

iteration #  238
Actual Output: 
 [[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]
Predicted Output: 
 [[0.30737699 0.38033063 0.31229237]
 [0.30737544 0.38032413 0.31230044]
 [0.30736624 0.38028274 0.31235102]
 [0.30737699 0.38033063 0.31229237]
 [0.3073762  0.38032728 0.31229652]
 [0.30737629 0.38032771 0.31229599]] 

cost: 
 1.103368561326939 

iteration #  239
Actual Output: 
 [[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]
Predicted Output: 
 [[0.35905011 0.28679078 0.35415912]
 [0.35904742 0.28678517 0.35416741]
 [0.35903112 0.28674952 0.35421936]
 [0.35905011 0.28679078 0.35415912]
 [0.35904873 0.28678789 0.35416338]
 [0.3590489  0.28678826 0.35416284]] 

cost: 
 1.103737165250413 

iteration #  240
Actual Output: 
 [[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]
Predicted Output: 
 [[0.3078873  0.37940673 0.31270597]
 [0.30788